In [3]:
import os
import glob
import pandas as pd
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import MinimalFCParameters

input_dir = "./processed"
output_dir = "./training_data"

def run_feature_engineering():
    print("Starting tsfresh feature engineering (Small Data Mode)...")
    os.makedirs(output_dir, exist_ok=True)

    # 1. Load the Target Labels
    labels_file = os.path.join(input_dir, "y_labels.csv")
    if not os.path.exists(labels_file):
        print("Could not find y_labels.csv!")
        return
    
    labels_df = pd.read_csv(labels_file)
    y = labels_df.set_index("Ride_ID")["Distraction_Score"]

    # This will extract only ~10 basic, powerful stats per sensor column
    extraction_settings = MinimalFCParameters()

    extracted_dataframes = []

    # 3. Extract features
    master_files = glob.glob(os.path.join(input_dir, "master_*.csv"))
    for file_path in master_files:
        sensor_name = os.path.basename(file_path).replace(".csv", "").replace("master_", "")
        print(f"Extracting minimal features for: {sensor_name.upper()}")
        
        df = pd.read_csv(file_path)
        
        features = extract_features(
            df, 
            column_id="Ride_ID", 
            column_sort="timestamp",
            default_fc_parameters=extraction_settings,
            n_jobs=0 
        )
        extracted_dataframes.append(features)

    # 4. Merge
    print("\nMerging data...")
    X_master = pd.concat(extracted_dataframes, axis=1)

    # 5. Clean up missing values
    impute(X_master)

    # CHANGE 3: We completely skip `select_features`!
    # We will trust our XGBoost model to handle the feature selection implicitly.
    X_selected = X_master 

    print(f"Total features kept for training: {X_selected.shape[1]}")

    # 6. Save the data and the feature list
    final_dataset_path = os.path.join(output_dir, "X_selected_features.csv")
    X_selected.to_csv(final_dataset_path)
    
    feature_list_path = os.path.join(output_dir, "selected_feature_names.txt")
    with open(feature_list_path, "w") as f:
        for feature_name in X_selected.columns:
            f.write(f"{feature_name}\n")

    print(f"\nSuccess, final dataset saved to: {final_dataset_path}")

if __name__ == "__main__":
    run_feature_engineering()

Starting tsfresh feature engineering (Small Data Mode)...
Extracting minimal features for: SENSOR


Feature Extraction: 100%|██████████| 12/12 [00:00<00:00, 683.14it/s]


Extracting minimal features for: BAROMETER


Feature Extraction: 100%|██████████| 12/12 [00:00<00:00, 682.97it/s]

Extracting minimal features for: PHOTOPLETHYSMOGRAPHY


...

Extracting minimal features for: GYROSCOPE


Feature Extraction: 100%|██████████| 36/36 [00:00<00:00, 732.83it/s]


Extracting minimal features for: ACCELEROMETER


Feature Extraction: 100%|██████████| 36/36 [00:00<00:00, 790.69it/s]


Merging data...
Total features kept for training: 150

Success, final dataset saved to: ./training_data/X_selected_features.csv
